# Notebook 3 · From behavioural summaries to a classifier

This notebook uses the lexical-decision data again, but at a new unit of analysis:
one row per participant. The aim is to practise the scikit-learn interface and locate
the points where leakage can enter an analysis, not to make a substantive claim about
language background from 21 people.

In [ ]:
from pathlib import Path

class Check:
    def _result(self, passed, success, hint):
        if passed:
            print(f"✅ {success}")
        else:
            print("✗ Not correct. Open the hint if needed.")
        return passed

    def equal(self, actual, expected, success="Correct.", hint="Value does not match the expected result."):
        try:
            passed = actual == expected
            if hasattr(passed, "all"):
                passed = bool(passed.all())
        except Exception:
            passed = False
        return self._result(bool(passed), success, hint)

    def shape(self, actual, expected, success="Shape is correct.", hint="Shape is incorrect."):
        return self._result(tuple(actual.shape) == tuple(expected), success, hint)

    def columns(self, frame, expected, success="Columns are correct.", hint="Inspect frame.columns and select with a list of names."):
        return self._result(list(frame.columns) == list(expected), success, hint)

    def choice(self, actual, expected, explanations):
        normalised = str(actual).strip().upper()
        hint = explanations.get(normalised, "Choose one of the listed letters.")
        return self._result(normalised == expected.upper(), "Correct.", hint)

check = Check()

## 1 · Load and prepare trial-level variables

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

candidates = [
    Path("book/data/real/lexical_decision.csv"),
    Path("../data/real/lexical_decision.csv"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Open this notebook from the workshop repository.")

trials = ...
trials["is_correct"] = ...
trials["RT_ms"] = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Load with `pd.read_csv`. Compare `Correct` with `"correct"`, and undo the natural logarithm in `RT` with `np.exp`.

</details>

In [ ]:
check.shape(trials, (1659, 30))
check.equal(int(trials["is_correct"].sum()), 1594)
check.equal(round(float(trials["RT_ms"].median()), 1), 570.0)

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
trials = pd.read_csv(data_path)
trials["is_correct"] = trials["Correct"].eq("correct")
trials["RT_ms"] = np.exp(trials["RT"])
```

</details>

## 2 · Make one row per participant

Create `participants` with `Subject` and `NativeLanguage`, plus mean reaction time,
accuracy, mean word frequency, and mean word length. Use named aggregation.

In [ ]:
participants = ...
participants.head()

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Group by `Subject` and `NativeLanguage` with `as_index=False`. Aggregate `RT_ms`, `is_correct`, `Frequency`, and `Length` with `.mean()`.

</details>

In [ ]:
check.shape(participants, (21, 6))
check.columns(participants, ["Subject", "NativeLanguage", "mean_rt_ms", "accuracy", "mean_frequency", "mean_length"])

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
participants = (
    trials.groupby(["Subject", "NativeLanguage"], as_index=False)
    .agg(
        mean_rt_ms=("RT_ms", "mean"),
        accuracy=("is_correct", "mean"),
        mean_frequency=("Frequency", "mean"),
        mean_length=("Length", "mean"),
    )
)
```

</details>

## 3 · Separate features and target

Let `X` contain the four numerical summaries and let `y` contain `NativeLanguage`.

In [ ]:
feature_names = ["mean_rt_ms", "accuracy", "mean_frequency", "mean_length"]
X = ...
y = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Select the feature list with square brackets. Select the target with one column name so that it remains a Series.

</details>

In [ ]:
check.shape(X, (21, 4))
check.shape(y, (21,))
check.equal(set(y), {"English", "Other"})

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
X = participants[feature_names]
y = participants["NativeLanguage"]
```

</details>

## 4 · Split before learning preprocessing parameters

Use 30% of participants as a test set. Set `random_state=42` and stratify by `y`.

In [ ]:
X_train, X_test, y_train, y_test = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Call `train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)`.

</details>

In [ ]:
check.shape(X_train, (14, 4))
check.shape(X_test, (7, 4))
check.equal(set(y_train), {"English", "Other"})

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
```

</details>

## 5 · Fit a pipeline

Create a pipeline containing `StandardScaler()` followed by
`LogisticRegression(max_iter=1000)`. Fit it on the training rows and predict the test
rows.

In [ ]:
model = ...
model.fit(...)
predictions = ...  # predict the test rows

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Use `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`; then call `.fit(X_train, y_train)` and `.predict(X_test)`.

</details>

In [ ]:
check.shape(predictions, (7,))
check.equal(set(predictions).issubset({"English", "Other"}), True)

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

</details>

## 6 · Inspect errors, not only a score

Create a 2 × 2 confusion matrix with labels ordered as `English`, `Other`. Then count
how many test predictions are correct.

In [ ]:
matrix = ...
n_correct_predictions = ...  # count matches
matrix

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Use `confusion_matrix(y_test, predictions, labels=["English", "Other"])`. Compare the two arrays and sum the Boolean results.

</details>

In [ ]:
check.shape(matrix, (2, 2))
check.equal(int(matrix.sum()), 7)
check.equal(n_correct_predictions, int((predictions == y_test).sum()))

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
matrix = confusion_matrix(
    y_test,
    predictions,
    labels=["English", "Other"],
)
n_correct_predictions = int((predictions == y_test).sum())
```

With only seven test participants, one changed prediction moves the accuracy by about
14 percentage points. Inspect the cases and sampling design rather than treating this
single split as a stable estimate.

</details>

### Reflection · What would count as leakage here?

Name two ways information from the test participants could accidentally affect model
training. What part of the pipeline protects against one of them?

In [ ]:
reflection_model = """
Leakage could occur if ...
The pipeline protects against ...
"""

<details class="notebook-reflection">
<summary><strong>Compare your reasoning</strong></summary>

Examples include scaling before the split, choosing features after examining test
performance, or allowing trials from the same participant to appear on both sides of a
trial-level split. The pipeline fits the scaler only on the training rows. It cannot
repair a split performed at the wrong unit or prevent decisions made after looking at
test results.

</details>